# 03 -- Plotly

## Objectifs pedagogiques

A la fin de ce notebook, vous saurez :

- creer des graphiques interactifs avec Plotly Express ;
- personnaliser les traces, layouts et annotations ;
- creer des graphiques 3D et des cartes geographiques ;
- utiliser les sous-graphiques avec `make_subplots` ;
- exporter en HTML et en image statique ;
- comprendre la difference entre Plotly Express et Plotly Graph Objects.

## Prerequis -- ce que vous devez maitriser

- les fonctions et type hints Python ;
- Pandas : `DataFrame`, filtrage ;
- Matplotlib et Seaborn (notions de base) ;

Ce que ce notebook **ne couvrira pas** :

- Dash (framework web) ;
- Altair (voir `04_Altair.ipynb`).

## Plan

1. Installation et premier graphique
2. Plotly Express -- l'API rapide
3. Scatter, Line, Bar, Histogram
4. Box, Violin, Strip
5. Heatmap et Contour
6. 3D : scatter3d, surface
7. Cartes geographiques (choropleth)
8. Sous-graphiques (make_subplots)
9. Animation (frames)
10. Plotly Graph Objects -- controle total
11. Export HTML et images
12. Synthese
13. Exercices

---

## 1. Installation et premier graphique

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

In [ ]:
df = px.data.iris()
fig = px.scatter(df, x='sepal_width', y='sepal_length', color='species')
fig.show()

Plotly genere du HTML/JS interactif : zoom, pan, hover, selection. Chaque graphique est autonome.

---

## 2. Plotly Express -- l'API rapide

Plotly Express est a Plotly ce que Seaborn est a Matplotlib : une API de haut niveau qui genere des graphiques complets en une ligne.

In [ ]:
# Donnees integrees
print(dir(px.data))

---

## 3. Scatter, Line, Bar, Histogram

### Scatter avec taille et couleur

In [ ]:
gapminder = px.data.gapminder()
fig = px.scatter(
    gapminder.query('year == 2007'),
    x='gdpPercap', y='lifeExp',
    size='pop', color='continent',
    hover_name='country', log_x=True,
    size_max=60,
    title='Gapminder 2007'
)
fig.show()

### Line

In [ ]:
france = gapminder.query('country == "France"')
fig = px.line(france, x='year', y='gdpPercap', title='PIB/hab France')
fig.show()

### Bar

In [ ]:
tips = px.data.tips()
fig = px.bar(tips, x='day', y='total_bill', color='sex', barmode='group', title='Additions par jour')
fig.show()

### Histogram

In [ ]:
fig = px.histogram(tips, x='total_bill', nbins=30, color='time', marginal='box', title='Distribution des additions')
fig.show()

---

## 4. Box, Violin, Strip

In [ ]:
fig = px.box(tips, x='day', y='total_bill', color='smoker', title='Boxplot')
fig.show()

In [ ]:
fig = px.violin(tips, x='day', y='total_bill', color='sex', box=True, title='Violinplot')
fig.show()

In [ ]:
fig = px.strip(tips, x='day', y='total_bill', color='smoker', title='Stripplot')
fig.show()

---

## 5. Heatmap et Contour

In [ ]:
iris = px.data.iris()
corr = iris.select_dtypes(include='number').corr()
fig = px.imshow(corr, text_auto='.2f', color_continuous_scale='RdBu_r', title='Correlation Iris')
fig.show()

---

## 6. 3D : scatter3d, surface

In [ ]:
fig = px.scatter_3d(iris, x='sepal_length', y='sepal_width', z='petal_length', color='species', title='Iris 3D')
fig.show()

In [ ]:
x = np.linspace(-5, 5, 50)
y = np.linspace(-5, 5, 50)
X, Y = np.meshgrid(x, y)
Z = np.sin(np.sqrt(X**2 + Y**2))

fig = go.Figure(data=[go.Surface(z=Z, x=X, y=Y)])
fig.update_layout(title='Surface sin(r)')
fig.show()

---

## 7. Cartes geographiques (choropleth)

In [ ]:
gm2007 = gapminder.query('year == 2007')
fig = px.choropleth(
    gm2007, locations='iso_alpha', color='lifeExp',
    hover_name='country', color_continuous_scale='Viridis',
    title='Esperance de vie 2007'
)
fig.show()

### Scatter sur carte

In [ ]:
fig = px.scatter_geo(
    gm2007, locations='iso_alpha', size='pop',
    color='continent', hover_name='country',
    projection='natural earth', title='Population 2007'
)
fig.show()

---

## 8. Sous-graphiques (make_subplots)

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Scatter', 'Histogram'))

fig.add_trace(go.Scatter(x=tips['total_bill'], y=tips['tip'], mode='markers'), row=1, col=1)
fig.add_trace(go.Histogram(x=tips['total_bill'], nbinsx=20), row=1, col=2)
fig.update_layout(title_text='Sous-graphiques', showlegend=False)
fig.show()

---

## 9. Animation (frames)

Plotly peut animer des graphiques en utilisant le parametre `animation_frame`.

In [ ]:
fig = px.scatter(
    gapminder, x='gdpPercap', y='lifeExp',
    animation_frame='year', animation_group='country',
    size='pop', color='continent', hover_name='country',
    log_x=True, size_max=55, range_x=[100, 100000], range_y=[25, 90],
    title='Gapminder anime'
)
fig.show()

---

## 10. Plotly Graph Objects -- controle total

Pour un controle fin, on utilise `go.Figure` et `go.Scatter`, `go.Bar`, etc.

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=[1, 2, 3, 4], y=[10, 11, 12, 13],
    mode='lines+markers', name='Serie A',
    line=dict(color='royalblue', width=2),
    marker=dict(size=8)
))
fig.add_trace(go.Bar(
    x=[1, 2, 3, 4], y=[5, 6, 7, 8],
    name='Serie B', marker_color='coral'
))
fig.update_layout(title='Graph Objects', xaxis_title='X', yaxis_title='Y')
fig.show()

### Annotations

In [ ]:
fig = px.scatter(iris, x='sepal_width', y='sepal_length', color='species')
fig.add_annotation(x=3.5, y=7.5, text='Zone interessante', showarrow=True, arrowhead=2)
fig.show()

---

## 11. Export HTML et images

In [ ]:
fig = px.scatter(iris, x='sepal_width', y='sepal_length', color='species')
fig.write_html('/tmp/plotly_example.html')
print('HTML exporte')

Pour exporter en PNG/SVG, installer `kaleido` : `pip install kaleido`.

In [ ]:
# fig.write_image('/tmp/plotly_example.png', scale=2)  # necessite kaleido

---

## Synthese

| Type | Plotly Express | Graph Objects |
|---|---|---|
| Scatter | `px.scatter` | `go.Scatter(mode='markers')` |
| Line | `px.line` | `go.Scatter(mode='lines')` |
| Bar | `px.bar` | `go.Bar` |
| Histogram | `px.histogram` | `go.Histogram` |
| Box | `px.box` | `go.Box` |
| Heatmap | `px.imshow` | `go.Heatmap` |
| 3D | `px.scatter_3d` | `go.Scatter3d` |
| Carte | `px.choropleth` | `go.Choropleth` |

### Regles a retenir

- **Plotly Express** pour explorer rapidement, **Graph Objects** pour le controle fin.
- **`fig.show()`** pour l'affichage interactif.
- **`fig.write_html()`** pour partager sans serveur.
- **`animation_frame`** pour des animations fluides.
- **`kaleido`** pour l'export en images statiques.

---

## Exercices

### Exercice 1 -- Gapminder anime

Reproduisez le graphique anime Gapminder avec les axes logarithmiques, taille proportionnelle a la population, et couleur par continent.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
gapminder = px.data.gapminder()
fig = px.scatter(
    gapminder, x='gdpPercap', y='lifeExp',
    animation_frame='year', animation_group='country',
    size='pop', color='continent', hover_name='country',
    log_x=True, size_max=55, range_x=[100, 100000], range_y=[25, 90]
)
fig.show()
```

</details>

### Exercice 2 -- Heatmap de correlation

Avec le dataset `tips`, creez une heatmap de correlation avec `px.imshow`, annotee et avec une echelle divergente.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
tips = px.data.tips()
corr = tips.select_dtypes(include='number').corr()
fig = px.imshow(corr, text_auto='.2f', color_continuous_scale='RdBu_r', title='Correlation Tips')
fig.show()
```

</details>

### Exercice 3 -- Sous-graphiques

Creez 4 sous-graphiques (2x2) montrant : scatter, histogram, box, violin pour le dataset `tips`.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from plotly.subplots import make_subplots
import plotly.graph_objects as go

tips = px.data.tips()
fig = make_subplots(rows=2, cols=2, subplot_titles=('Scatter', 'Histogram', 'Box', 'Violin'))
fig.add_trace(go.Scatter(x=tips['total_bill'], y=tips['tip'], mode='markers'), row=1, col=1)
fig.add_trace(go.Histogram(x=tips['total_bill']), row=1, col=2)
fig.add_trace(go.Box(y=tips['total_bill'], name='total_bill'), row=2, col=1)
fig.add_trace(go.Violin(y=tips['total_bill'], name='total_bill'), row=2, col=2)
fig.update_layout(height=600, showlegend=False)
fig.show()
```

</details>

### Exercice 4 -- Carte choropleth

Affichez l'esperance de vie en 2007 sur une carte choropleth avec une echelle de couleurs `Plasma`.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
gapminder = px.data.gapminder()
gm2007 = gapminder.query('year == 2007')
fig = px.choropleth(
    gm2007, locations='iso_alpha', color='lifeExp',
    hover_name='country', color_continuous_scale='Plasma',
    title='Esperance de vie 2007'
)
fig.show()
```

</details>

---

## Ressources externes

- [Documentation Plotly](https://plotly.com/python/)
- [Plotly Express](https://plotly.com/python/plotly-express/)
- [Plotly Graph Objects](https://plotly.com/python/graph-objects/)
- [Plotly Gallery](https://plotly.com/python/basic-charts/)